# B100 Intelligence - Exploratory Data Analysis
**Bluestock Fintech Capstone Project**

In [1]:
import psycopg2, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

conn = psycopg2.connect(host='localhost', port=5432, dbname='bluestock_dw', user='tanishqyadav')
def r(q):
    with conn.cursor() as c:
        c.execute(q)
        return pd.DataFrame(c.fetchall(), columns=[d[0] for d in c.description])
print('Connected to bluestock_dw!')
companies = r('SELECT * FROM dim_company')
print(f'Total companies: {len(companies)}')
companies[['symbol','company_name','roce_pct','roe_pct']].head(10)

Connected to bluestock_dw!
Total companies: 92


,symbol,company_name,roce_pct,roe_pct
0,ABB,Abbott India Ltd,46.0,34.9
1,ADANIENSOL,Adani Energy Solutions Ltd,9.0,8.59
2,ADANIENT,Adani Enterprises Ltd,11.6,13.64
3,ADANIGREEN,Adani Green Energy Ltd,96.5,14.7
4,ADANIPORTS,Adani Ports & Special Economic Zone Ltd,12.9,18.1
5,ADANIPOWER,Adani Power Ltd,32.2,57.1
6,AMBUJACEM,Ambuja Cements Ltd,12.8,9.24
7,APOLLOHOSP,Apollo Hospitals\nChain of Indian private hosp...,17.2,13.81
8,ASIANPAINT,Asian Paints\nIndian Multi-National Paint and ...,41.75,31.45
9,ATGL,Adani Total Gas Ltd,21.2,20.5


## Health Label Distribution

In [2]:
scores = r('SELECT * FROM fact_ml_scores')
scores['total_score'] = pd.to_numeric(scores['total_score'])
print(scores['health_label'].value_counts())
print('Top 5:')
print(scores.nlargest(5,'total_score')[['symbol','total_score','health_label']])

health_label
Average      43
Good         29
Excellent    21
Weak          5
Poor          2
Name: count, dtype: int64
Top 5:
        symbol  total_score health_label
8   ASIANPAINT          100    Excellent
11  BAJAJ-AUTO          100    Excellent
25   COALINDIA          100    Excellent
48        INFY          100    Excellent
66   NESTLEIND          100    Excellent


## Revenue Trends

In [3]:
pl = r("SELECT * FROM fact_profit_loss WHERE year_label != 'TTM'")
pl['sales'] = pd.to_numeric(pl['sales'], errors='coerce')
pl['net_profit'] = pd.to_numeric(pl['net_profit'], errors='coerce')
top5 = scores.nlargest(5,'total_score')['symbol'].tolist()
print('Top 5 companies:', top5)
for sym in top5:
    d = pl[pl['symbol']==sym]
    print(f'{sym}: {len(d)} years of data')

Top 5 companies: ['ASIANPAINT', 'BAJAJ-AUTO', 'COALINDIA', 'INFY', 'NESTLEIND']
ASIANPAINT: 12 years of data
BAJAJ-AUTO: 12 years of data
COALINDIA: 12 years of data
INFY: 12 years of data
NESTLEIND: 12 years of data


## Debt Analysis

In [4]:
bs = r('SELECT * FROM fact_balance_sheet')
bs['debt_to_equity'] = pd.to_numeric(bs['debt_to_equity'], errors='coerce')
latest = bs.sort_values('year_label').groupby('symbol').last().reset_index()
low_debt = latest[latest['debt_to_equity'] < 1].dropna(subset=['debt_to_equity'])
print(f'Companies with D/E < 1: {len(low_debt)}')
print(low_debt[['symbol','debt_to_equity']].sort_values('debt_to_equity').head(10))
conn.close()
print('EDA Complete!')

Companies with D/E < 1: 66
        symbol  debt_to_equity
76     SBILIFE        0.000000
58        LICI        0.000000
54      JIOFIN        0.000000
27    DIVISLAB        0.000583
13  BAJAJHLDNG        0.001161
63      MARUTI        0.001390
19    BOSCHLTD        0.002434
44     ICICIGI        0.002868
52         ITC        0.004067
65      NAUKRI        0.006126
EDA Complete!
